# Multi-provider calls, sampling controls, and temperature
**The messages format · temperature, top_p, max_tokens, stop · switching provider by editing .env**



## 1. Setup — three clients, one function

A **provider** is a service that gives us access to AI models, while a model is the AI system that performs the task.

In this session, we use three options:

- Groq: Provides access to hosted AI models.
- Gemini: Provides access to Google's AI models.
- Ollama: Allows us to run supported models locally on our computer.

We can use the same general client interface to communicate with different providers. Usually, we need to change the client configuration and model name.

To use a hosted provider, we may need an API key stored securely in our environment configuration.

In [ ]:
import sys; sys.path[:0] = [".", "module01", "../module01"]
import os
from providers_utils import ask, classify_ticket, gemini_client, groq_client, list_gemini_models, ollama_client, pick_gemini_model
from tickets_data import ALL_TICKETS, BORDERLINE_TICKETS, CLEAR_TICKETS

groq = groq_client()
GROQ_MODEL = os.getenv("MODEL")

try:
    gemini = gemini_client()
    # Gemini's Flash model name changes often - verify instead of trusting a name written down months ago.
    flash_models = list_gemini_models()
    GEMINI_MODEL = os.getenv("GEMINI_MODEL") or pick_gemini_model(flash_models)
    print("Gemini models available:", flash_models)
    print("using:", GEMINI_MODEL)
except Exception as e:
    gemini, GEMINI_MODEL = None, None
    print("Gemini not configured yet:", e)

try:
    ollama = ollama_client()
    OLLAMA_MODEL = "llama3.2:3b"
except Exception as e:
    ollama = None
    print("Ollama not available:", e)


Gemini models available: ['gemini-2.5-flash', 'gemini-2.5-flash-lite', 'gemini-3-flash-preview', 'gemini-3.1-flash-lite', 'gemini-3.1-flash-lite-preview', 'gemini-3.5-flash', 'gemini-3.5-flash-lite', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'gemini-flash-latest', 'gemini-flash-lite-latest']
using: gemini-3.8-flash


## 2. The messages format, properly
An LLM receives messages in a structured format. Each message usually contains a role and some content.

The three common roles are:

- System: Gives instructions about how the AI should behave.

- User: Contains the question or request.

- Assistant: Contains the AI's response.

In [ ]:
system = "You are a terse assistant for a retail bank. Answer in one short sentence."
user = "What is an EMI?"

with_system = ask(groq, GROQ_MODEL, system, user)
without_system = ask(groq, GROQ_MODEL, "", user)

print("WITH system message:\n ", with_system["text"])
print("\nWITHOUT a system message:\n ", without_system["text"])


WITH system message:
  EMI is the fixed monthly payment you make to repay a loan.

WITHOUT a system message:
  **


## 3. Same prompt, three providers

I can send the same question to Groq, Gemini, and Ollama and compare their responses.

Example question: "What is the difference between a workflow and an agent?"

I should compare:

- Answer quality: Is the answer correct and useful?
- Token usage: How many tokens were used for the input and output?
- Speed: How quickly does the model respond?
- Cost: How much does using the model cost?
- Reliability: Does the model produce consistent and valid responses?



In [ ]:
prompt = "In one sentence, what is the difference between a workflow and an agent?"
results = {}
for label, client, model in [("groq", groq, GROQ_MODEL), ("gemini", gemini, GEMINI_MODEL), ("ollama", ollama, "llama3.2:3b" if ollama else None)]:
    if client is None or model is None:
        print(f"--- {label}: skipped (not configured)\n"); continue
    try:
        r = ask(client, model, "You are a concise AI engineering tutor.", prompt)
        results[label] = r
        print(f"--- {label} ({model})")
        print(" ", r["text"].strip())
        print(f"   tokens: prompt={r['prompt_tokens']} completion={r['completion_tokens']} total={r['total_tokens']}\n")
    except Exception as e:
        print(f"--- {label}: error ({type(e).__name__}: {str(e)[:100]})\n")

--- groq (openai/gpt-oss-20b)
  A workflow is a fixed, pre‑defined sequence of tasks, while an agent is an autonomous entity that can decide, execute, and adapt tasks—often orchestrating or generating workflows on the fly.
   tokens: prompt=97 completion=139 total=236

--- gemini (gemini-3.8-flash)
  A workflow follows a predefined
   tokens: prompt=25 completion=5 total=221

--- ollama (llama3.2:3b)
  A workflow is a series of tasks or processes that are executed in a specific order, whereas an agent is a software entity that autonomously interacts with the environment, making decisions and taking actions to achieve a goal.
   tokens: prompt=48 completion=43 total=91



## 4. Sampling: temperature, top_p, max_tokens, stop

**A. Temperature**
Temperature controls the randomness of the model's output.
- Low temperature: Usually produces more predictable responses.
- High temperature: Allows more variation in responses.

For example, when classifying a customer complaint, I generally want a consistent label rather than a different answer each time.

A temperature of 0 aims for more consistent output, but it does not guarantee identical results across every model and provider.

**B. Top-p**
Top-p controls the group of possible next tokens from which the model can choose.
For example, a lower top-p restricts the model to a smaller group of likely tokens.
Temperature and top-p both affect token selection. In most experiments, I should adjust one at a time so I can understand its effect.

**C. Max Tokens**
Max tokens sets a limit on the number of tokens the model can generate in its response.
For example:

- A very low limit may cut off the answer.
- A higher limit gives the model more room to complete its response.
If the limit is too low, the model may return an incomplete or even empty response.

**D. Stop**
A stop sequence tells the model to stop generating when a specified text sequence appears.

For example, using a newline as a stop sequence can help end a response at the first line break, depending on the provider's implementation.

**Main idea:** These settings help control the length, variation, and stopping point of an AI response.

In [ ]:
short = ask(groq, GROQ_MODEL, "You are a helpful assistant.", "Explain what a vector database is.", max_tokens=15)
enough = ask(groq, GROQ_MODEL, "You are a helpful assistant.", "Explain what a vector database is.", max_tokens=150)
print("max_tokens=15:\n ", short["text"], "\n")
print("max_tokens=150:\n ", enough["text"][:300])


max_tokens=15:
   

max_tokens=150:
  ## What is a Vector Database?




In [ ]:
r = ask(groq, GROQ_MODEL, "You are a helpful assistant. Answer in a short paragraph.",
        "List three benefits of retrieval-augmented generation.", max_tokens=200, stop=["\n"])
print(r["text"])


Retrieval‑augmented generation (RAG) boosts a model’s usefulness by (1) grounding responses in real documents, which cuts hallucinations and improves factual accuracy; (2) allowing the system to pull in up‑to‑date or domain‑specific information that the base model may not know, keeping answers current and relevant; and (3) reducing the need for massive pre‑training, since the retrieval component supplies the bulk of the knowledge, making the approach more efficient and easier to adapt to new topics.


## 5. The temperature sweep — where it actually matters

In this experiment, I will classify customer support tickets using different temperature values.

The model may classify a ticket into categories such as billing or technical support.

I will test these temperature values:
- 0.0
- 0.3
- 0.7
- 1.0

I will run the same ticket three times at each temperature and compare the labels.

**What are clear and borderline tickets?**

**Clear tickets:** The problem is easy to identify.

Example: "I was charged twice for the same payment."

**Borderline tickets:** The problem could reasonably belong to more than one category.

Example: "My payment failed, and now I cannot access my account."

**What am I checking?**

I am checking whether the model gives the same label every time.
- If all three labels match, the result is stable.
- If the labels differ, the result has drifted.

For an important production routing system, I would generally start with temperature 0 and test its reliability. I would also validate the output and measure classification accuracy using test data.

**Important:** Low temperature alone does not guarantee that a classification system is correct. The prompt, model, and quality of the test data also matter.

In [ ]:
TEMPERATURES = [0.0, 0.3, 0.7, 1.0]
RUNS_PER_TEMPERATURE = 3

if groq is None:
    print("skipped (no Groq client)")
else:
    for tid, text in BORDERLINE_TICKETS:
        print(f"\n=== {tid} ===\n{text}\n")
        for temp in TEMPERATURES:
            labels = []
            for _ in range(RUNS_PER_TEMPERATURE):
                result = classify_ticket(groq, GROQ_MODEL, text, temperature=temp)
                labels.append(result["label"] if result["valid"] else f"INVALID:{result['label']!r}")
            stable = len(set(labels)) == 1
            print(f"  temperature={temp:<4} {'stable' if stable else 'DRIFTED':8} {labels}")



=== borderline_1 ===
I upgraded my plan but I'm still being charged the old price and the new features aren't showing up either.

  temperature=0.0  stable   ["INVALID:''", "INVALID:''", "INVALID:''"]
  temperature=0.3  stable   ["INVALID:''", "INVALID:''", "INVALID:''"]
  temperature=0.7  stable   ["INVALID:''", "INVALID:''", "INVALID:''"]
  temperature=1.0  stable   ["INVALID:''", "INVALID:''", "INVALID:''"]

=== borderline_2 ===
Your app keeps logging me out and now I can't tell if I'm still being billed for the plan I signed up for.

  temperature=0.0  stable   ["INVALID:''", "INVALID:''", "INVALID:''"]
  temperature=0.3  stable   ["INVALID:''", "INVALID:''", "INVALID:''"]
  temperature=0.7  stable   ["INVALID:''", "INVALID:''", "INVALID:''"]
  temperature=1.0  stable   ["INVALID:''", "INVALID:''", "INVALID:''"]

=== borderline_3 ===
I'm not sure who to ask, but something seems wrong with my account since the last update.

  temperature=0.0  stable   ["INVALID:''", "INVALID:''", "

## 6. Switching provider — three lines, no code change

One useful feature of a well-designed application is that it can work with different providers without changing all its code.

The .env file may contain settings such as **BASE_URL**, **API_KEY**, **MODEL** The exact variable names depend on how the project is configured.

To switch providers, I update the relevant configuration values and restart the notebook kernel if needed. Then I rerun the setup cells and compare the results.


In [ ]:
# After editing .env, restart the kernel (Kernel > Restart), then re-run section 1's setup cell
# and this cell to see the SAME code now talking to a different provider.
import importlib, providers_utils
importlib.reload(providers_utils)
from providers_utils import ask
print("BASE_URL now:", os.getenv("BASE_URL"))
print("MODEL now:", os.getenv("MODEL"))

BASE_URL now: https://api.groq.com/openai/v1
MODEL now: openai/gpt-oss-20b


## 7. Reflection

Answer questions 1–4 above, one or two lines each.

1.
2.
3.
4.

Then run the checkpoint from the terminal: `uv run pytest tests/test_s11.py`